<a href="https://colab.research.google.com/github/Thanjaivalavan/M2_GenAI_AgenticAI/blob/main/04_van_toy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Slides 71-72: Variational Autoencoder (VAE) — ELBO objective
------------------------------------------------------------
Implements, from scratch (numpy only)

    ELBO:  L(theta,phi;x) = E_q(z|x)[log p(x|z)] - D_KL(q(z|x) || p(z))

    i) Reconstruction term: E_q(z|x)[log p(x|z)]
       -> approximated here with Mean Squared Error (as the slide notes
          MSE is typical for continuous values).

    ii) KL term: D_KL(q(z|x) || p(z))  with q(z|x)=N(mu,sigma^2), p(z)=N(0,1)
        Closed form:
            D_KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)

    Encoder q_phi(z|x): maps x -> (mu, log_var) of a Gaussian latent.
    Decoder p_theta(x|z): reconstructs x from a sampled z.
    Reparameterization trick: z = mu + sigma * epsilon, epsilon ~ N(0,1)
    (lets gradients flow through the sampling step).

Toy task: learn to encode/decode 2-D points on a noisy circle.
"""

import numpy as np

np.random.seed(0)

DATA_DIM = 2
LATENT_DIM = 2
HIDDEN_DIM = 16


def sample_data(n):
    """Toy dataset: points on a noisy circle of radius 3."""
    theta = np.random.uniform(0, 2 * np.pi, n)
    r = 3.0 + np.random.normal(0, 0.1, n)
    x = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1)
    return x


class VAE:
    """Single hidden layer encoder + decoder, all in numpy with manual grads."""

    def __init__(self):
        # Encoder: x -> hidden -> (mu, log_var)
        self.We1 = np.random.randn(DATA_DIM, HIDDEN_DIM) * 0.3
        self.be1 = np.zeros(HIDDEN_DIM)
        self.W_mu = np.random.randn(HIDDEN_DIM, LATENT_DIM) * 0.3
        self.b_mu = np.zeros(LATENT_DIM)
        self.W_logvar = np.random.randn(HIDDEN_DIM, LATENT_DIM) * 0.3
        self.b_logvar = np.zeros(LATENT_DIM)

        # Decoder: z -> hidden -> x_reconstructed
        self.Wd1 = np.random.randn(LATENT_DIM, HIDDEN_DIM) * 0.3
        self.bd1 = np.zeros(HIDDEN_DIM)
        self.Wd2 = np.random.randn(HIDDEN_DIM, DATA_DIM) * 0.3
        self.bd2 = np.zeros(DATA_DIM)

    def encode(self, x):
        h = np.tanh(x @ self.We1 + self.be1)
        mu = h @ self.W_mu + self.b_mu
        log_var = h @ self.W_logvar + self.b_logvar
        return h, mu, log_var

    def reparameterize(self, mu, log_var):
        """z = mu + sigma * epsilon   (reparameterization trick)"""
        std = np.exp(0.5 * log_var)
        eps = np.random.randn(*mu.shape)
        z = mu + std * eps
        return z, std, eps

    def decode(self, z):
        h = np.tanh(z @ self.Wd1 + self.bd1)
        x_hat = h @ self.Wd2 + self.bd2
        return h, x_hat

    def train_step(self, x, lr):
        n = x.shape[0]
        h_enc, mu, log_var = self.encode(x)
        z, std, eps = self.reparameterize(mu, log_var)
        h_dec, x_hat = self.decode(z)

        # --- Losses (per the ELBO formula) ---
        recon_loss = np.mean(np.sum((x - x_hat) ** 2, axis=1))          # (i) reconstruction (MSE)
        kl_loss = -0.5 * np.mean(np.sum(1 + log_var - mu**2 - np.exp(log_var), axis=1))  # (ii) KL term
        total_loss = recon_loss + kl_loss   # ELBO = -(recon - KL) minimized as (recon + KL)

        # --- Backprop (manual, chain rule through the reparameterization trick) ---
        d_xhat = -2 * (x - x_hat) / n                     # d(recon_loss)/d(x_hat)
        d_h_dec = (d_xhat @ self.Wd2.T) * (1 - h_dec ** 2)

        dWd2 = h_dec.T @ d_xhat
        dbd2 = d_xhat.sum(axis=0)
        dWd1 = z.T @ d_h_dec
        dbd1 = d_h_dec.sum(axis=0)
        d_z = d_h_dec @ self.Wd1.T

        # z = mu + std*eps  ->  dz/dmu = 1 ;  dz/dstd = eps ; std = exp(0.5*logvar)
        d_mu_from_recon = d_z
        d_std = d_z * eps
        d_logvar_from_recon = d_std * 0.5 * std

        # KL gradients (closed form derivative of -0.5*sum(1+logvar-mu^2-exp(logvar)))
        d_mu_from_kl = mu / n
        d_logvar_from_kl = 0.5 * (np.exp(log_var) - 1) / n

        d_mu = d_mu_from_recon + d_mu_from_kl
        d_logvar = d_logvar_from_recon + d_logvar_from_kl

        dW_mu = h_enc.T @ d_mu
        db_mu = d_mu.sum(axis=0)
        dW_logvar = h_enc.T @ d_logvar
        db_logvar = d_logvar.sum(axis=0)

        d_h_enc = (d_mu @ self.W_mu.T + d_logvar @ self.W_logvar.T) * (1 - h_enc ** 2)
        dWe1 = x.T @ d_h_enc
        dbe1 = d_h_enc.sum(axis=0)

        # --- Gradient descent updates ---
        self.Wd2 -= lr * dWd2; self.bd2 -= lr * dbd2
        self.Wd1 -= lr * dWd1; self.bd1 -= lr * dbd1
        self.W_mu -= lr * dW_mu; self.b_mu -= lr * db_mu
        self.W_logvar -= lr * dW_logvar; self.b_logvar -= lr * db_logvar
        self.We1 -= lr * dWe1; self.be1 -= lr * dbe1

        return recon_loss, kl_loss, total_loss


def main():
    print("Toy data: 2-D points on a noisy circle (radius ~3)")
    vae = VAE()
    lr = 0.01
    epochs = 4000

    for epoch in range(epochs):
        x = sample_data(64)
        recon_loss, kl_loss, total_loss = vae.train_step(x, lr)
        if epoch % 500 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:4d} | recon={recon_loss:.4f}  KL={kl_loss:.4f}  ELBO_loss={total_loss:.4f}")

    print("\n--- Reconstruction check ---")
    x_test = sample_data(5)
    _, mu, log_var = vae.encode(x_test)
    z, _, _ = vae.reparameterize(mu, log_var)
    _, x_hat = vae.decode(z)
    for orig, rec in zip(x_test, x_hat):
        print(f"original=({orig[0]:.2f},{orig[1]:.2f})  reconstructed=({rec[0]:.2f},{rec[1]:.2f})")

    print("\n--- Generation: sample z ~ N(0,1) and decode ---")
    z_new = np.random.randn(5, LATENT_DIM)
    _, x_gen = vae.decode(z_new)
    for g in x_gen:
        r = np.sqrt(g[0]**2 + g[1]**2)
        print(f"generated=({g[0]:.2f},{g[1]:.2f})  radius={r:.2f}  (target radius ~3.0)")


if __name__ == "__main__":
    main()


Toy data: 2-D points on a noisy circle (radius ~3)
epoch    0 | recon=8.1041  KL=1.0516  ELBO_loss=9.1557
epoch  500 | recon=1.0287  KL=2.0664  ELBO_loss=3.0951
epoch 1000 | recon=0.8835  KL=1.9903  ELBO_loss=2.8738
epoch 1500 | recon=0.7571  KL=2.0003  ELBO_loss=2.7574
epoch 2000 | recon=0.9978  KL=2.0027  ELBO_loss=3.0006
epoch 2500 | recon=0.8590  KL=1.9373  ELBO_loss=2.7963
epoch 3000 | recon=0.6866  KL=1.9838  ELBO_loss=2.6704
epoch 3500 | recon=0.7734  KL=1.9521  ELBO_loss=2.7255
epoch 3999 | recon=0.6946  KL=1.9876  ELBO_loss=2.6821

--- Reconstruction check ---
original=(2.86,0.17)  reconstructed=(2.99,0.31)
original=(-2.14,-2.05)  reconstructed=(-1.78,-2.28)
original=(-1.59,-2.56)  reconstructed=(-1.64,-2.43)
original=(1.86,2.18)  reconstructed=(1.27,2.74)
original=(-0.77,-2.97)  reconstructed=(-0.64,-2.58)

--- Generation: sample z ~ N(0,1) and decode ---
generated=(2.51,-0.89)  radius=2.67  (target radius ~3.0)
generated=(-2.20,0.67)  radius=2.30  (target radius ~3.0)
genera